In [ ]:

import sqlite3
import pandas as pd
from IPython.display import display, HTML
import ipywidgets as widgets
from pathlib import Path

# Caminho absoluto robusto para o banco de dados, mesmo no Voilà
project_root = Path.cwd().parents[1]
db_path = project_root / "data" / "sjur_recortes.db"

if not db_path.exists():
    raise FileNotFoundError(f"❌ Banco de dados não encontrado: {db_path}")


In [ ]:

def carregar_tabela(nome_tabela):
    with sqlite3.connect(db_path) as conn:
        total = pd.read_sql_query(f"SELECT COUNT(*) as total FROM {nome_tabela}", conn).iloc[0]['total']
        df = pd.read_sql_query(f"SELECT * FROM {nome_tabela} LIMIT 50", conn)
    return total, df


In [ ]:

abas = widgets.Tab()
titulos = ["emails_processados", "publicacoes", "partes", "metadados"]
conteudos = []

for nome in titulos:
    try:
        total, df = carregar_tabela(nome)
        html = f"<h4>📊 Total de registros: {total}</h4>"
        out = widgets.Output()
        with out:
            display(HTML(html))
            display(df)
        conteudos.append(out)
    except Exception as e:
        out = widgets.Output()
        with out:
            display(HTML(f"<b style='color:red;'>Erro ao carregar {nome}: {e}</b>"))
        conteudos.append(out)

abas.children = conteudos
for i, t in enumerate(titulos):
    abas.set_title(i, t)

display(HTML("<h2>📂 Visualização Interativa das Tabelas do Pipeline</h2>"))
display(abas)
